# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [ ]:
from class_dataset import SegmentationClassDataset, num_classes as class_num_classes
from instance_dataset import SegmentationInstanceDataset

mode = "instance"
# mode = "class"

if mode == "instance":
    dataset_class = SegmentationInstanceDataset
    num_classes = 2
    model_path = "model_instance.pth"
elif mode == "class":
    dataset_class = SegmentationClassDataset
    num_classes = class_num_classes
    model_path = "model_class.pth"

In [ ]:
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import utils

import torch


model = get_model_instance_segmentation(num_classes)

dataset = dataset_class("../../training_data", get_transforms(is_train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

## Training and Validation


In [ ]:
import csv
from engine import evaluate, train_one_epoch
from sklearn.model_selection import KFold


k_folds = 5
num_epochs = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")

kfold = KFold(n_splits=k_folds, shuffle=True)

# use our dataset and defined transformations
full_dataset = dataset_class("../../training_data", get_transforms(is_train=True))

results = [
    [
        "fold",
        "epoch",
        "batch",
        "loss_classifier",
        "loss_box_reg",
        "loss_objectness",
        "loss_rpn_box_reg",
    ]
]

for fold, (train_ids, test_ids) in enumerate(kfold.split(full_dataset)):
    print(f"FOLD {fold}")
    print("-" * 20)

    train_dataset = torch.utils.data.Subset(full_dataset, train_ids)
    train_dataset.dataset.transforms = get_transforms(is_train=True)

    valid_dataset = torch.utils.data.Subset(full_dataset, test_ids)
    valid_dataset.dataset.transforms = get_transforms(is_train=False)

    # define training and validation data loaders
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=4, shuffle=True, collate_fn=utils.collate_fn
    )
    validation_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=1, shuffle=False, collate_fn=utils.collate_fn
    )

    model = get_model_instance_segmentation(num_classes).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        training_losses = train_one_epoch(
            model, optimizer, train_loader, device, epoch, print_freq=10
        )

        for batch, losses in enumerate(training_losses):
            results.append(
                [
                    fold,
                    epoch,
                    batch,
                    losses["loss_classifier"],
                    losses["loss_box_reg"],
                    losses["loss_objectness"],
                    losses["loss_rpn_box_reg"],
                ]
            )

        # update the learning rate
        lr_scheduler.step()
        # evaluate on the validation dataset
        evaluate(model, validation_loader, device=device)

# save results to a CSV file
with open("results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

In [ ]:
# Train final model on all data

print(f"FINAL TRAINING")
print("-" * 20)

# define training and validation data loaders
train_loader = torch.utils.data.DataLoader(
    full_dataset, batch_size=1, shuffle=True, collate_fn=utils.collate_fn
)

model = get_model_instance_segmentation(num_classes).to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

results = [
    [
        "epoch",
        "batch",
        "loss_classifier",
        "loss_box_reg",
        "loss_objectness",
        "loss_rpn_box_reg",
    ]
]

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    training_losses = train_one_epoch(
        model, optimizer, train_loader, device, epoch, print_freq=10
    )

    for batch, losses in enumerate(training_losses):
        results.append(
            [
                epoch,
                batch,
                losses["loss_classifier"],
                losses["loss_box_reg"],
                losses["loss_objectness"],
                losses["loss_rpn_box_reg"],
            ]
        )

    # update the learning rate
    lr_scheduler.step()

# save results to a CSV file
with open("final-results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

# save the model
torch.save(model.state_dict(), model_path)